# Exercise: Sigma-Matrix, Emittance, 1-sigma and 2-sigma Ellipse
<html>
<div class="alert alert-info" role="alert" style="margin-top: 10px">
The goal of this exercise is to write methods that evaluate the $\sigma$-matrix and emittance of a bunch. Furthermore,
we want to evaluate and plot the $1\sigma$ and $2\sigma$ ellipse of the particle distribution.
</div>
</html>

In [1]:
from pyH5root import SDDSParser as parser
#from pyH5root.Plotter import *
from pyH5root import SDDSPlotter as plotter
import numpy as np

from matplotlib import rc
import matplotlib.pylab as plt

ModuleNotFoundError: No module named 'SDDSParser'

## Statistics Opverview Plot


In [ ]:
fns = []
fns.append('/Users/adelmann/Desktop/v1.9/RFphotoinjector-1.9.stat')
# fns.append('/Users/adelmann/Desktop/v1.6/RFphotoinjector-1.6.stat')

s = parser.SDDSParser()
s.parse(fns[0])
p = plotter.SDDSPlotter()


In [ ]:
# just some nicer output
np.set_printoptions(formatter={'float': '{: 0.4f}'.format})

sigma = getMoments(bunch.particles)

print ( sigma )

## Emittance
Using the function above you're now able to compute the emittance. Please complete.

In [ ]:
# -------------------------------------------------------------------
# compute the emittance
# @param particles (6 x N)-dimensional matrix.
# The state is given by (x, x', y, y', z, z')
# @returns an array of dimension (3 x 1) with emittance (ex, ey, ez)
# -------------------------------------------------------------------
def getEmittance(particles):
    emit = np.zeros( shape = (1, 3) )
        
    sigma = getMoments(particles)
        
    for i in range(3):
        j = 2 * i
        emit[:, i] = np.sqrt(sigma[j, j] * 
                             sigma[j + 1, j + 1] - 
                             sigma[j, j + 1] ** 2)
        
    return emit

In [ ]:
emit = getEmittance(bunch.particles)

print ( emit )

## 1-sigma and 2-sigma Ellipse

In [ ]:
from matplotlib.patches import Ellipse
import numpy as np

# Obtain the s-sigma ellipse
# @param xvar is first particle variable
# @param yvar is second particle variable
# @param s is 1, 2, ... - sigma
def getEllipse(xvar, yvar, s):
    
    x = np.asarray(xvar)
    y = np.asarray(yvar)
    
    mean = [np.mean(x), np.mean(y)]
    
    sigma = np.cov(x, y)
    
    # get eigenvalues lam and eigenvectors v
    lam, v = np.linalg.eig(sigma)
    
    l1 = np.sqrt( lam[0] )
    l2 = np.sqrt( lam[1] )
    rot = np.rad2deg(np.arccos(v[0, 0]))
    
    # width and height need to be multiplied by 2 since they are diameters
    ell = Ellipse(xy=(mean[0], mean[1]), width=l1 * s * 2,  height=l2 * s * 2, angle=rot)
    ell.set_alpha(0.6)
    
    return ell

In [ ]:
import matplotlib.pyplot as plt

# Plot the phase space (xvar, yvar) and its s-sigma ellipse
# @param xvar is first particle variable
# @param xlab is the x-axis label
# @param yvar is second particle variable
# @param ylab is the y-axis label
# @param s is 1, 2, ... - sigma
def plot_ellipse(xvar, xlab, yvar, ylab, s):
    ellipse = getEllipse(xvar, yvar, s)

    fig = plt.figure(dpi=300)
    fig.set_size_inches(9,9)
    ax = fig.add_subplot(111)
    ellipse.set_label(str(s) + '-sigma')
    ax.add_artist(ellipse)
    ax.scatter(x=xvar, y=yvar, s=2, marker='.', cmap=plt.cm.viridis)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.legend(handles=[ellipse], loc='upper left')
    plt.show()


plot_ellipse(bunch['x'], 'x [mm]', bunch["x'"], "x' [mrad]", s = 1)